# Vision Encoder — 2K Colab render

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lroe/vision-encoder/blob/main/render_2k_colab.ipynb)

This notebook renders the **18:33 Vision Encoder explainer** at **2560 × 1440 (2K)** and writes the result to **Google Drive**.

- **Access first:** step 1 mounts Drive before anything is written. Output goes to `MyDrive/VisionEncoder/`.
- **2K only:** frames are captured at exactly 2560 × 1440; the still test and the final ffprobe both assert it.
- **Resumable:** every 300-frame chunk (10 s at 30 fps) is copied to Drive. If Colab disconnects, rerun the notebook; completed chunks are skipped.
- **Live progress:** the render cell prints frame counts, chunk completions, per-worker fps and a running ETA.
- **2 CPU Colab:** two workers, one ffmpeg thread each. Set `CONCURRENCY = 2` in step 1. Free Colab can take several hours; chunks make it safe.
- **Source:** https://github.com/lroe/vision-encoder

> The source is a **live Canvas 2D explainer**, not a Remotion project. This notebook adapts the reference 2K Colab workflow to frame-by-frame Playwright capture of that canvas. There is no other renderer to install.

> **First run:** `Runtime → Run all`, approve the Google Drive prompt, and let the render + stitch cells run. Interrupt the render cell to stop; finished chunks are already on Drive.



In [ ]:
# ---------------------------------------------------------------------------
# 1) Settings
# ---------------------------------------------------------------------------
REPO_URL = "https://github.com/lroe/vision-encoder.git"
BRANCH = "main"
DRIVE_FOLDER = "VisionEncoder"

TARGET_WIDTH = 2560          # 2K UHD
TARGET_HEIGHT = 1440
FPS = 30
CHUNK_FRAMES = 300           # 10 s at 30 fps; copied to Drive after each chunk
CONCURRENCY = 2              # Colab: 2 workers / 2 cores
CRF = 18                     # lower = better quality / larger file
PORT = 8000

EXPECTED_DURATION = 1113.6   # verified after clone against audio.wav
OUTPUT_NAME = f"vision-encoder-{TARGET_WIDTH}x{TARGET_HEIGHT}.mp4"
SCALE = TARGET_WIDTH / 1920

FORCE_RENDER = False         # True = ignore an existing final MP4
FORCE_STITCH = False         # True = re-stitch even if the final MP4 exists
DELETE_CHUNKS = False        # only after the final MP4 is verified

import os
from pathlib import Path

assert TARGET_WIDTH % 2 == 0 and TARGET_HEIGHT % 2 == 0, "Video dimensions must be even."
assert abs(TARGET_HEIGHT - 1080 * SCALE) < 0.01, "Keep the 16:9 aspect ratio."

REPO_DIR = Path("/content/vision-encoder")
WORK_DIR = Path("/content/render-work")
DRIVE_ROOT = Path("/content/drive/MyDrive") / DRIVE_FOLDER
CHUNKS_DIR = DRIVE_ROOT / "chunks"
STILLS_DIR = DRIVE_ROOT / "stills"
LOGS_DIR = DRIVE_ROOT / "logs"
CONFIG_PATH = WORK_DIR / "render_config.json"
SERVER_PATH = WORK_DIR / "serve_static.py"
WORKER_PATH = WORK_DIR / "render_worker.py"
SILENT_WAV = WORK_DIR / "silence.wav"

print(f"Target       : {TARGET_WIDTH} × {TARGET_HEIGHT} (scale {SCALE:.6f})")
print(f"Frame rate   : {FPS} fps")
print(f"Chunk size   : {CHUNK_FRAMES} frames")
print(f"Concurrency  : {CONCURRENCY}")
print(f"Drive output : {DRIVE_ROOT}")
print(f"Local work   : {WORK_DIR}")



## 1 · Google Drive access

This is the only place the notebook asks for permission. The next cell opens Google's authorization dialog. Nothing is written to Drive until you approve it.

Files will live in `MyDrive/VisionEncoder/`:

```
VisionEncoder/
├── vision-encoder-2560x1440.mp4
├── chunks/                 # resumable 2K segments
├── stills/                 # 2K test frame
├── logs/                   # renderer event log
└── render_metadata.json    # final ffprobe + settings
```



In [ ]:
from google.colab import drive
import shutil

MOUNT = Path("/content/drive")
if (MOUNT / "MyDrive").exists():
    print("Google Drive is already mounted.")
else:
    print("Requesting Google Drive access...")
    drive.mount(str(MOUNT), force_remount=False)
    print("Access granted.")

assert (MOUNT / "MyDrive").exists(), "Google Drive was not mounted."

DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
for sub in (CHUNKS_DIR, STILLS_DIR, LOGS_DIR):
    sub.mkdir(parents=True, exist_ok=True)

local = shutil.disk_usage("/content")
print(f"\nLocal disk free : {local.free / 1e9:.1f} GB")
try:
    remote = shutil.disk_usage(DRIVE_ROOT)
    print(f"Drive space free: {remote.free / 1e9:.1f} GB")
except OSError:
    pass
print(f"Output folder   : {DRIVE_ROOT}")
print("Drive is ready.")



## 2 · Install Playwright, Chromium and ffmpeg

Colab usually has ffmpeg and git. This cell installs the Python Playwright client, Pillow, tqdm, and the Chromium build with its system libraries. It can take a few minutes on the first run.



In [ ]:
import os
import subprocess
import sys


def run_stream(cmd, cwd=None, check=True, env=None):
    printable = " ".join(str(x) for x in cmd)
    print(f"$ {printable}", flush=True)
    merged_env = {**os.environ, "DEBIAN_FRONTEND": "noninteractive", **(env or {})}
    proc = subprocess.Popen(
        [str(x) for x in cmd],
        cwd=cwd,
        env=merged_env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    for line in proc.stdout:
        print(line, end="", flush=True)
    code = proc.wait()
    if check and code != 0:
        raise RuntimeError(f"Command failed with exit code {code}: {printable}")
    return code


try:
    subprocess.check_output(["ffmpeg", "-version"], text=True)
    print("ffmpeg already installed.")
except Exception:
    run_stream(["apt-get", "update", "-qq"])
    run_stream(["apt-get", "install", "-y", "-qq", "ffmpeg", "git", "curl", "ca-certificates"])

print("\nInstalling Python packages...")
run_stream([sys.executable, "-m", "pip", "install", "-q", "playwright", "pillow", "tqdm"])

print("\nInstalling Chromium for Playwright (with system dependencies)...")
run_stream([sys.executable, "-m", "playwright", "install", "--with-deps", "chromium"])

import playwright
from PIL import Image
import tqdm

print("\nPython :", sys.version.split()[0])
print("Playwright:", getattr(playwright, "__version__", "installed"))
print("CPU count:", os.cpu_count())



## 3 · Clone the good video from GitHub

This always checks out the latest `main`. It clones the good base video, not any local experiment. The narration WAV is about 51 MB, so the first clone can take a moment.



In [ ]:
import json
import subprocess

print("Fetching the latest project from GitHub...")
if (REPO_DIR / ".git").exists():
    run_stream(["git", "-C", str(REPO_DIR), "fetch", "--depth", "1", "origin", BRANCH])
    run_stream(["git", "-C", str(REPO_DIR), "reset", "--hard", f"origin/{BRANCH}"])
else:
    if REPO_DIR.exists():
        import shutil as _shutil
        _shutil.rmtree(REPO_DIR)
    run_stream(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(REPO_DIR)])

commit = subprocess.check_output(
    ["git", "-C", str(REPO_DIR), "log", "-1", "--format=%h  %ad  %s", "--date=iso"],
    text=True,
).strip()
print("\nChecked out:", commit)

assert (REPO_DIR / "preview" / "index.html").exists(), "preview/index.html is missing."
assert (REPO_DIR / "preview" / "js" / "app.js").exists(), "preview/js/app.js is missing."
assert (REPO_DIR / "preview" / "js" / "scenes" / "index.js").exists(), "scene entry is missing."
audio = REPO_DIR / "audio.wav"
assert audio.exists(), "audio.wav is missing."
audio_mb = audio.stat().st_size / 1e6
assert audio_mb > 40, "audio.wav is missing or truncated."

duration = float(subprocess.check_output([
    "ffprobe", "-v", "error", "-show_entries", "format=duration",
    "-of", "default=noprint_wrappers=1:nokey=1", str(audio),
], text=True).strip())
TOTAL_FRAMES = int(round(duration * FPS))
if abs(duration - EXPECTED_DURATION) > 0.1:
    print(
        f"Note: audio is {duration:.3f} s, expected {EXPECTED_DURATION:.3f} s. "
        "Rendering the full actual duration."
    )

WORK_DIR.mkdir(parents=True, exist_ok=True)
CHUNKS_DIR.mkdir(parents=True, exist_ok=True)
STILLS_DIR.mkdir(parents=True, exist_ok=True)
LOGS_DIR.mkdir(parents=True, exist_ok=True)

chunks = []
for index, start in enumerate(range(0, TOTAL_FRAMES, CHUNK_FRAMES)):
    end = min(TOTAL_FRAMES, start + CHUNK_FRAMES)
    chunks.append({"index": index, "start": start, "end": end})

CONFIG = {
    "repo_dir": str(REPO_DIR),
    "work_dir": str(WORK_DIR),
    "drive_chunks": str(CHUNKS_DIR),
    "silent_wav": str(SILENT_WAV),
    "width": TARGET_WIDTH,
    "height": TARGET_HEIGHT,
    "fps": FPS,
    "crf": CRF,
    "concurrency": CONCURRENCY,
    "total_frames": TOTAL_FRAMES,
    "duration_seconds": duration,
    "url": f"http://127.0.0.1:{PORT}/preview/?capture=1&t=0&scale={SCALE:.17g}",
    "chunks": chunks,
}
CONFIG_PATH.write_text(json.dumps(CONFIG, indent=2) + "\n", encoding="utf-8")

print(f"Narration     : {audio_mb:.1f} MB, {duration:.3f} s")
print(f"Total frames  : {TOTAL_FRAMES:,} at {FPS} fps")
print(f"Chunks        : {len(chunks)} × up to {CHUNK_FRAMES} frames")
print(f"2K URL        : {CONFIG['url']}")
print("Project is ready.")



## 4 · Runtime helpers

The notebook writes its own tiny static server and the frame-capture worker here, so the GitHub repo stays clean. Nothing is downloaded or installed in this step.



In [ ]:
# Static file server with correct MIME types for .js, .mjs, .woff2, .wav
SERVER_CODE = r'''import sys
from functools import partial
from http.server import SimpleHTTPRequestHandler, ThreadingHTTPServer


class Handler(SimpleHTTPRequestHandler):
    extensions_map = {
        **SimpleHTTPRequestHandler.extensions_map,
        ".html": "text/html; charset=utf-8",
        ".js": "text/javascript; charset=utf-8",
        ".mjs": "text/javascript; charset=utf-8",
        ".css": "text/css; charset=utf-8",
        ".json": "application/json; charset=utf-8",
        ".wav": "audio/wav",
        ".woff2": "font/woff2",
        ".png": "image/png",
        ".jpg": "image/jpeg",
        ".svg": "image/svg+xml",
    }

    def log_message(self, *args):
        pass


if __name__ == "__main__":
    root = sys.argv[1]
    port = int(sys.argv[2])
    handler = partial(Handler, directory=root)
    ThreadingHTTPServer(("127.0.0.1", port), handler).serve_forever()
'''

# One Playwright worker: capture frames, encode a chunk, copy it to Drive
WORKER_CODE = r'''import base64
import json
import os
import shutil
import subprocess
import sys
import time
import traceback
from pathlib import Path

from playwright.sync_api import sync_playwright

CFG_PATH = Path(os.environ.get("RENDER_CONFIG", "/content/render-work/render_config.json"))
cfg = json.loads(CFG_PATH.read_text())
WORKER = int(sys.argv[1])

if WORKER >= cfg["concurrency"]:
    raise SystemExit(f"worker {WORKER} is outside concurrency {cfg['concurrency']}")

FRAME_JS = (
    "(t) => {"
    "  window.__drawAt(t);"
    "  const c = document.getElementById('stage');"
    "  return c.toDataURL('image/jpeg', 0.96).split(',')[1];"
    "}"
)


def log(kind, **kw):
    print("@@" + kind + "@@ " + json.dumps(kw), flush=True)


def main():
    chunks = [c for c in cfg["chunks"] if c["index"] % cfg["concurrency"] == WORKER]
    drive_chunks = Path(cfg["drive_chunks"])
    work_dir = Path(cfg["work_dir"])
    drive_chunks.mkdir(parents=True, exist_ok=True)
    work_dir.mkdir(parents=True, exist_ok=True)

    with sync_playwright() as p:
        browser = p.chromium.launch(
            headless=True,
            args=[
                "--no-sandbox",
                "--disable-dev-shm-usage",
                "--disable-background-timer-throttling",
                "--disable-renderer-backgrounding",
            ],
        )
        context = browser.new_context(
            viewport={"width": 1280, "height": 720},
            device_scale_factor=1,
        )
        page = context.new_page()
        page.route(
            "**/audio.wav",
            lambda route: route.fulfill(
                path=cfg["silent_wav"], content_type="audio/wav"
            ),
        )
        page.set_default_timeout(120000)
        page.goto(cfg["url"], wait_until="networkidle", timeout=120000)
        page.wait_for_function(
            "() => window.__state && window.__state().ready", timeout=60000
        )

        size = page.evaluate(
            "() => { const c = document.getElementById('stage'); return [c.width, c.height]; }"
        )
        if size != [cfg["width"], cfg["height"]]:
            log("ERROR", worker=WORKER, message=f"canvas size {size}, expected 2K")
            raise SystemExit(2)

        log("READY", worker=WORKER, size=size, chunks=len(chunks))

        for chunk in chunks:
            idx = chunk["index"]
            start = chunk["start"]
            end = chunk["end"]
            frames = end - start
            final_path = drive_chunks / f"chunk_{idx:04d}.mp4"

            if final_path.exists() and final_path.stat().st_size > 10_000:
                log("SKIP", worker=WORKER, index=idx, frames=frames)
                continue

            local_path = work_dir / f"chunk_{idx:04d}.mp4"
            ff = subprocess.Popen(
                [
                    "ffmpeg", "-y", "-loglevel", "error",
                    "-f", "image2pipe",
                    "-framerate", str(cfg["fps"]),
                    "-vcodec", "mjpeg",
                    "-i", "-",
                    "-an",
                    "-c:v", "libx264",
                    "-preset", "veryfast",
                    "-crf", str(cfg["crf"]),
                    "-pix_fmt", "yuv420p",
                    "-threads", "1",
                    "-r", str(cfg["fps"]),
                    str(local_path),
                ],
                stdin=subprocess.PIPE,
            )

            started = time.time()
            done = 0
            try:
                for frame in range(start, end):
                    t = frame / cfg["fps"]
                    b64 = page.evaluate(FRAME_JS, t)
                    ff.stdin.write(base64.b64decode(b64))
                    done += 1
                    elapsed = max(0.001, time.time() - started)
                    log(
                        "FRAME",
                        worker=WORKER,
                        index=idx,
                        frame=frame,
                        fps=round(done / elapsed, 2),
                    )
                ff.stdin.close()
                rc = ff.wait()
                if rc != 0:
                    log("ERROR", worker=WORKER, index=idx, message=f"ffmpeg exit {rc}")
                    raise SystemExit(3)
            except BaseException:
                try:
                    ff.kill()
                except Exception:
                    pass
                raise

            if not local_path.exists() or local_path.stat().st_size < 10_000:
                log("ERROR", worker=WORKER, index=idx, message="chunk is empty")
                raise SystemExit(4)

            tmp_path = drive_chunks / f"chunk_{idx:04d}.part.mp4"
            shutil.copy2(local_path, tmp_path)
            os.replace(tmp_path, final_path)
            local_path.unlink(missing_ok=True)
            log(
                "CHUNK",
                worker=WORKER,
                index=idx,
                frames=frames,
                seconds=round(time.time() - started, 2),
                size_mb=round(final_path.stat().st_size / 1e6, 2),
            )

        browser.close()
    log("DONE", worker=WORKER)


if __name__ == "__main__":
    try:
        main()
    except SystemExit:
        raise
    except BaseException:
        log("ERROR", worker=WORKER, message=traceback.format_exc())
        raise
'''

print(f'Helper scripts ready: server {len(SERVER_CODE)} chars, worker {len(WORKER_CODE)} chars')



## 5 · Start the static server and verify a 2K still

This renders one frame at 2560 × 1440, checks the actual pixel dimensions, saves it to Drive, and shows it inline. If this looks right, the full render will too.



In [ ]:
import base64
import socket
import time
import wave
from urllib.request import urlopen

# ---- static server ---------------------------------------------------------
SERVER_PATH.write_text(SERVER_CODE, encoding="utf-8")


def port_open(port):
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        s.settimeout(0.3)
        return s.connect_ex(("127.0.0.1", port)) == 0


if not port_open(PORT):
    print(f"Starting static server on 127.0.0.1:{PORT}...")
    server_proc = subprocess.Popen(
        [sys.executable, str(SERVER_PATH), str(REPO_DIR), str(PORT)],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
        start_new_session=True,
    )
    for _ in range(80):
        if port_open(PORT):
            break
        time.sleep(0.25)
    else:
        raise RuntimeError("Static server did not start.")
else:
    print(f"Static server already running on {PORT}.")

assert urlopen(f"http://127.0.0.1:{PORT}/preview/").status == 200

# ---- a tiny silent WAV so the page boots instantly -------------------------
with wave.open(str(SILENT_WAV), "wb") as w:
    w.setnchannels(1)
    w.setsampwidth(2)
    w.setframerate(24000)
    w.writeframes(b"\x00\x00" * 12000)

# ---- write the worker for the long render ---------------------------------
WORKER_PATH.write_text(WORKER_CODE, encoding="utf-8")
print("Worker written:", WORKER_PATH)

# ---- 2K still --------------------------------------------------------------
from PIL import Image
from IPython.display import Image as IPImage, display
from playwright.sync_api import sync_playwright

FRAME_JS = (
    "(t) => {"
    "  window.__drawAt(t);"
    "  const c = document.getElementById('stage');"
    "  return c.toDataURL('image/jpeg', 0.96).split(',')[1];"
    "}"
)
STILL_FRAME = 390  # 13 s: the 10-trillion moment
still_local = WORK_DIR / f"still_{STILL_FRAME:05d}.jpg"
still_drive = STILLS_DIR / f"still_{STILL_FRAME:05d}_{TARGET_WIDTH}x{TARGET_HEIGHT}.jpg"

print(f"\nRendering still frame {STILL_FRAME} at {TARGET_WIDTH} × {TARGET_HEIGHT}...")
with sync_playwright() as p:
    browser = p.chromium.launch(
        headless=True,
        args=["--no-sandbox", "--disable-dev-shm-usage"],
    )
    context = browser.new_context(
        viewport={"width": 1280, "height": 720},
        device_scale_factor=1,
    )
    page = context.new_page()
    page.route(
        "**/audio.wav",
        lambda route: route.fulfill(
            path=str(SILENT_WAV), content_type="audio/wav"
        ),
    )
    page.goto(CONFIG["url"], wait_until="networkidle", timeout=120000)
    page.wait_for_function(
        "() => window.__state && window.__state().ready", timeout=60000
    )
    size = page.evaluate(
        "() => { const c = document.getElementById('stage'); return [c.width, c.height]; }"
    )
    assert size == [TARGET_WIDTH, TARGET_HEIGHT], f"Canvas is {size}, expected 2K."
    b64 = page.evaluate(FRAME_JS, STILL_FRAME / FPS)
    still_local.write_bytes(base64.b64decode(b64))
    browser.close()

im = Image.open(still_local)
print(f"Rendered size: {im.width} × {im.height}")
assert (im.width, im.height) == (TARGET_WIDTH, TARGET_HEIGHT), "2K still has the wrong dimensions."

still_drive.write_bytes(still_local.read_bytes())
print(f"Saved to Drive: {still_drive}")
display(IPImage(filename=str(still_drive), width=960))



## 6 · Render the full video with live progress

Two Playwright workers capture 30 fps frames at 2K. Each 300-frame chunk is encoded with x264, copied to Drive, and then the next chunk starts.

- **Stop safely:** interrupt this cell. Completed chunks stay on Drive.
- **Resume:** rerun the notebook from the top (or just the render cell). Existing chunks are skipped automatically.
- **Time:** 2 CPU Colab is roughly 2–6 hours for the full 18:33 at 2K. The exact speed depends on the runtime; the bar and per-worker fps are live.



In [ ]:
import json
import os
import queue
import shutil
import signal
import subprocess
import sys
import threading
import time
from pathlib import Path

from IPython.display import FileLink, display
from tqdm.auto import tqdm

if not FORCE_RENDER and (DRIVE_ROOT / OUTPUT_NAME).exists():
    print("Final MP4 already exists:")
    display(FileLink(str(DRIVE_ROOT / OUTPUT_NAME)))
    print("Delete the chunk files first, then set FORCE_RENDER = True to re-render.")
else:
    log_path = WORK_DIR / f"render_events_{time.strftime('%Y%m%d_%H%M%S')}.jsonl"
    log_handle = open(log_path, "w", encoding="utf-8")
    print(f"Event log: {log_path} (copied to Drive when the render stops)")

    procs = []
    q = queue.Queue()
    errors = []
    processed = 0
    started = time.time()

    def reader(proc, worker):
        for line in proc.stdout:
            q.put((worker, line.rstrip("\n")))
        q.put((worker, None))

    for worker in range(CONCURRENCY):
        proc = subprocess.Popen(
            [sys.executable, "-u", str(WORKER_PATH), str(worker)],
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
            start_new_session=True,
        )
        procs.append(proc)
        threading.Thread(target=reader, args=(proc, worker), daemon=True).start()

    bar = tqdm(total=TOTAL_FRAMES, unit=" frame", dynamic_ncols=True)
    active = len(procs)
    try:
        while active:
            worker, line = q.get()
            if line is None:
                active -= 1
                continue
            if not line:
                continue
            log_handle.write(line + "\n")
            log_handle.flush()

            if line.startswith("@@FRAME@@ "):
                event = json.loads(line[len("@@FRAME@@ "):])
                processed += 1
                bar.update(1)
                bar.set_postfix(
                    w=event.get("worker"),
                    fps=event.get("fps"),
                    chunk=event.get("index"),
                )
            elif line.startswith("@@SKIP@@ "):
                event = json.loads(line[len("@@SKIP@@ "):])
                processed += event.get("frames", 0)
                bar.update(event.get("frames", 0))
                print(f"  worker {event.get('worker')} · chunk {event.get('index')} already on Drive")
            elif line.startswith("@@CHUNK@@ "):
                event = json.loads(line[len("@@CHUNK@@ "):])
                print(
                    f"  chunk {event.get('index'):04d} done · "
                    f"{event.get('frames')} frames · {event.get('seconds')}s · "
                    f"{event.get('size_mb')} MB · worker {event.get('worker')}"
                )
            elif line.startswith("@@READY@@ "):
                event = json.loads(line[len("@@READY@@ "):])
                print(f"  worker {event.get('worker')} ready · canvas {event.get('size')}")
            elif line.startswith("@@ERROR@@ "):
                event = json.loads(line[len("@@ERROR@@ "):])
                errors.append(event)
                print(f"  ERROR worker {event.get('worker')}: {event.get('message')}")
            else:
                print(line, flush=True)
    except KeyboardInterrupt:
        print("\nInterrupted. Terminating workers; completed chunks are safe on Drive.")
        for proc in procs:
            if proc.poll() is None:
                try:
                    os.killpg(os.getpgid(proc.pid), signal.SIGTERM)
                except Exception:
                    proc.terminate()
        for proc in procs:
            try:
                proc.wait(timeout=10)
            except Exception:
                proc.kill()
    finally:
        bar.close()
        log_handle.close()

    try:
        shutil.copy2(log_path, LOGS_DIR / log_path.name)
        print(f"Event log saved: {LOGS_DIR / log_path.name}")
    except Exception as exc:
        print(f"Could not copy the event log to Drive: {exc}")

    elapsed = time.time() - started
    missing = [
        c["index"] for c in CONFIG["chunks"]
        if not (CHUNKS_DIR / f"chunk_{c['index']:04d}.mp4").exists()
    ]
    print(f"\nProcessed {processed:,} / {TOTAL_FRAMES:,} frames in {elapsed/60:.1f} min.")
    if missing:
        print(f"Missing chunks: {missing}")
        print("Rerun this cell to fill only the missing chunks.")
    else:
        print("All chunks are on Drive. Run the stitch cell next.")
    if errors:
        print("Worker errors:")
        for e in errors:
            print(" ", e)
        raise RuntimeError("One or more workers failed; chunks so far are safe on Drive.")



## 7 · Stitch the 2K master and mux the narration

This concatenates the chunks, adds `audio.wav`, writes the final MP4 to Drive, verifies 2560 × 1440 with ffprobe, and shows a download link.



In [ ]:
import json
import os
import shutil
import subprocess
from pathlib import Path

from IPython.display import FileLink, display

final_local = WORK_DIR / OUTPUT_NAME
final_drive = DRIVE_ROOT / OUTPUT_NAME
concat_path = WORK_DIR / "concat.txt"

missing = [
    c["index"] for c in CONFIG["chunks"]
    if not (CHUNKS_DIR / f"chunk_{c['index']:04d}.mp4").exists()
]
assert not missing, f"Missing chunks: {missing}. Run the render cell first."

lines = []
for chunk in CONFIG["chunks"]:
    p = CHUNKS_DIR / f"chunk_{chunk['index']:04d}.mp4"
    lines.append("file '" + str(p).replace("'", "'\\''") + "'")
concat_path.write_text("\n".join(lines) + "\n", encoding="utf-8")

def run_stream(cmd, check=True):
    printable = " ".join(str(x) for x in cmd)
    print(f"$ {printable}", flush=True)
    proc = subprocess.Popen(
        [str(x) for x in cmd],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    for line in proc.stdout:
        print(line, end="", flush=True)
    code = proc.wait()
    if check and code != 0:
        raise RuntimeError(f"Command failed with exit code {code}")
    return code

if final_drive.exists() and not FORCE_STITCH and not FORCE_RENDER:
    print("Final MP4 already exists; verifying it.")
else:
    run_stream([
        "ffmpeg", "-y",
        "-f", "concat", "-safe", "0", "-i", str(concat_path),
        "-i", str(REPO_DIR / "audio.wav"),
        "-map", "0:v:0", "-map", "1:a:0",
        "-c:v", "copy",
        "-c:a", "aac", "-b:a", "192k",
        "-shortest",
        "-movflags", "+faststart",
        str(final_local),
    ])
    tmp = DRIVE_ROOT / (OUTPUT_NAME + ".part.mp4")
    shutil.copy2(final_local, tmp)
    os.replace(tmp, final_drive)
    print(f"\nSaved to Drive: {final_drive}")

probe = json.loads(subprocess.check_output([
    "ffprobe", "-v", "error",
    "-select_streams", "v:0",
    "-show_entries", "stream=width,height,r_frame_rate",
    "-show_entries", "format=duration,size",
    "-of", "json", str(final_drive),
], text=True))
stream = probe["streams"][0]
width = int(stream["width"])
height = int(stream["height"])
duration = float(probe["format"]["duration"])
fps_text = stream.get("r_frame_rate", "0/1")
num, den = fps_text.split("/")
fps_value = float(num) / float(den)

print(f"\nFinal: {width} × {height}, {fps_value:.3f} fps, {duration:.2f} s")
print(f"Size : {int(probe['format']['size']) / 1e9:.2f} GB")
assert (width, height) == (TARGET_WIDTH, TARGET_HEIGHT), "Final is not 2K."
assert abs(duration - EXPECTED_DURATION) < 1.0, "Final duration is off by more than 1 s."

metadata = {
    "source_repo": REPO_URL,
    "commit": subprocess.check_output(
        ["git", "-C", str(REPO_DIR), "log", "-1", "--format=%H"], text=True
    ).strip(),
    "width": width,
    "height": height,
    "fps": FPS,
    "duration_seconds": duration,
    "chunk_frames": CHUNK_FRAMES,
    "concurrency": CONCURRENCY,
    "crf": CRF,
    "total_frames": TOTAL_FRAMES,
    "file_size_bytes": int(probe["format"]["size"]),
}
(DRIVE_ROOT / "render_metadata.json").write_text(
    json.dumps(metadata, indent=2) + "\n", encoding="utf-8"
)

display(FileLink(str(final_drive)))



## 8 · Optional housekeeping

The final MP4 is independent of the chunk files. Once you have downloaded and played the final file, you can delete `chunks/` to free Drive space. Set `DELETE_CHUNKS = True` in step 1, then rerun this cell.



In [ ]:
import subprocess

verify = DRIVE_ROOT / OUTPUT_NAME
if verify.exists():
    print(f"Final file: {verify}")
    print(f"Size      : {verify.stat().st_size / 1e9:.2f} GB")
    print(subprocess.check_output([
        "ffprobe", "-v", "error", "-select_streams", "v:0",
        "-show_entries", "stream=width,height,r_frame_rate",
        "-of", "default=noprint_wrappers=1", str(verify),
    ], text=True).strip())
else:
    print("Final file not found. Run the render and stitch cells first.")

if DELETE_CHUNKS and verify.exists():
    chunks = sorted(CHUNKS_DIR.glob("chunk_*.mp4"))
    freed = sum(p.stat().st_size for p in chunks)
    for p in chunks:
        p.unlink()
    print(f"Deleted {len(chunks)} chunk files ({freed / 1e9:.2f} GB freed).")
elif DELETE_CHUNKS:
    print("Refusing to delete chunks because the final MP4 is missing.")



## Troubleshooting

| Symptom | Fix |
| --- | --- |
| `Google Drive was not mounted` | Rerun step 1 and approve the browser popup. Allow popups for `colab.research.google.com`. |
| Playwright Chromium will not start | Rerun step 2. The `--with-deps` install adds the shared libraries Colab needs. |
| `Canvas is … expected 2K` | The page did not receive `scale=1.333…`. Check that `preview/js/app.js` is the current file and that the server is serving `/preview/`. |
| A chunk looks black or wrong | Open the still in `MyDrive/VisionEncoder/stills/` first. If the still is good, delete the bad chunk from `chunks/` and rerun the render cell. |
| Render stops when Colab disconnects | Rerun the notebook. Finished chunks are checked on Drive and skipped. |
| Drive runs out of space | Keep `DELETE_CHUNKS = False` until the final file is downloaded, then set it to `True`; or lower `CHUNK_FRAMES`. |
| Want a smaller file | Raise `CRF` in step 1 (20–22), delete the old `chunks/`, and set `FORCE_RENDER = True`. |
| Text looks soft | The source is vector at 2K; the only lossy step is JPEG quality 0.96 before x264. You can raise `quality` in the worker and use a lower CRF for a larger master. |

The renderer is **not Remotion** for this project. It captures the deterministic Canvas frames from the GitHub source at `t = frame / 30`, encodes 10-second chunks with x264, and muxes the original narration at the end. The still test proves 2560 × 1440 before the long render.

